# Extraction de Keywords

## Imports

In [17]:
import os, re, random
import yake
from collections import Counter

## Extraire les mots clés d'un document avec Yake

https://github.com/INESCTEC/yake

In [10]:
# Instantier l'extracteur de mots clés
kw_extractor = yake.KeywordExtractor(lan="fr", top=50)
kw_extractor

In [8]:
# Charger all.txt
data_path = "../data"
if not os.path.exists(data_path):
    data_path = "../../data"

corpus = open(os.path.join(data_path, "all.txt"), 'r', encoding='utf-8').read()
len(corpus)

299699167

In [18]:
jours = "lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche"
mois = "janvier|f.vrier|mars|avril|mai|juin|juillet|ao.t|septembre|octobre|novembre|d.cembre"   # le point remplace une lettre accentuée

def positions_des_dates(annee):
    motif = "(" + jours + r")\s+\d+\s+(" + mois + r")\s+" + str(annee)
    return [m.start() for m in re.finditer(motif, corpus, re.I)]

dates_1923 = positions_des_dates(1923)
len(dates_1923)

91

In [19]:
milieu = dates_1923[len(dates_1923) // 2]
dates_1922 = [p for p in positions_des_dates(1922) if p < milieu]
dates_1924 = [p for p in positions_des_dates(1924) if p > milieu]

debut = min(p for p in dates_1923 if not dates_1922 or p > dates_1922[-1])
fin = dates_1924[0] if dates_1924 else len(corpus)
year_text = corpus[debut:fin]
print(len(year_text), "caractères")

# Vérifier : le début et la fin doivent être du texte de 1923
print(year_text[:300])
print("...")
print(year_text[-300:])

# Sauvegarder l'année dans un fichier : les notebooks s2, s3 et s4 le réutiliseront
tmp_path = os.path.join(data_path, "tmp")
if not os.path.exists(tmp_path):
    os.mkdir(tmp_path)
with open(os.path.join(tmp_path, "1923.txt"), 'w', encoding='utf-8') as f:
    f.write(year_text)

3626897 caractères
MARDI 16 JANVIER 1923 RESULTATS : 24C0i 25491 25495 25530 27246 27330 27457 27827' 23101 3Ô353 I - Tyro* > ( a ' M 3 JuI e Jier-^^Tiirre 32473 24133 35312 35400 35552 35259 37311 33763 38810 28873 Premier^^otUe-)^■ i Tteoerary' Wk&rd) *' *** mS0 41329 43444 46574 49622. 50981 51230 53573 54937 55192
...
lars. Une première souscription a-rapporté une somme de il,410 dollars. BOXE CONGRES DE LA FEDERATION INTERNATIONALE DES AMATEURS Le congrès interuat-cnal a continué hier après-midi au Sporüug Club. Jeux. Olympiques de 1924. — Le congrès prévoit la participation de 200 athlètes. Voici le programme: 


In [20]:
CHUNK = 100_000
chunks = [year_text[i:i + CHUNK] for i in range(0, len(year_text), CHUNK)]
len(chunks)

37

In [21]:
# Récupérer le texte du premier bloc
text = chunks[0]
text[:500]

"MARDI 16 JANVIER 1923 RESULTATS : 24C0i 25491 25495 25530 27246 27330 27457 27827' 23101 3Ô353 I - Tyro* > ( a ' M 3 JuI e Jier-^^Tiirre 32473 24133 35312 35400 35552 35259 37311 33763 38810 28873 Premier^^otUe-)^■ i Tteoerary' Wk&rd) *' *** mS0 41329 43444 46574 49622. 50981 51230 53573 54937 55192 J ^ a D } ao ^ Ti a ucoun ^/fdr Vantkr* Tariia- 5/14/ 5718J 53172 64275 C2/Ô1 63834 644S5 64941 67066 67245 * - - ?- aoûb -:. 4-iaucqurt, teaux. _4«ntai xa*na- 68546 70415 7060/ 71518 71759 73446 735"

In [22]:
# Extraire les mots clés de ce texte
keywords = kw_extractor.extract_keywords(text)
keywords

[('JANVIER', 0.006997052138718618),
 ('Roi', 0.010648164613909012),
 ('Port', 0.011052027365912626),
 ('grand', 0.011836063986772827),
 ('D’UN', 0.01191735295823918),
 ('d’une', 0.012215574141122992),
 ('faut', 0.012820097024092054),
 ('variole', 0.014127069052883202),
 ('faire', 0.014792099226568718),
 ('C’est', 0.015132370354948467),
 ('Bruxelles', 0.015648276819281028),
 ('JuI e Jier', 0.016942389827759894),
 ('qu’il', 0.017279072311074027),
 ('jour', 0.01758835306117529),
 ('Fédération', 0.017884165894724066),
 ('travaux', 0.017957940900875866),
 ('Lloyd George', 0.018620695440710506),
 ('Fédération française', 0.018759026748986482),
 ('Salon', 0.018796529440820747),
 ('Terre', 0.02003867068378937),
 ('guerre', 0.020841100690894676),
 ('Belgique', 0.021062112352449088),
 ('maturité', 0.02106703549359082),
 ('pts', 0.021784424434730887),
 ('cours', 0.021828361560148973),
 ('Société', 0.02223102509310947),
 ('qualité', 0.022732340698492757),
 ('Majesté', 0.023183053190626062),
 ('var

In [23]:
# Ne garder que les bigrammes
kept = []
for kw, score in keywords:
    words = kw.split()
    if len(words) == 2:
        kept.append(kw)
kept

['Lloyd George',
 'Fédération française',
 'bonne qualité',
 'd’un grand',
 'D’une part']

## Faire la même opération sur tous les documents

In [26]:
IGNORER = {"agence", "rossel", "demande", "dem", "bon", "bons", "bonne", "bonnes",
           "jeune", "jeunes", "fille", "filles", "homme", "maison", "louer", "vendre",
           "belle", "gages", "rue", "van", "prix", "ouvriers", "ouvrier", "fem",
           "place", "salaire", "salaires", "faire", "offres", "mais"}

N_BLOCS = 20
random.seed(1923)
compteur = Counter()
for n, block in enumerate(random.sample(chunks, min(N_BLOCS, len(chunks))), 1):
    keywords = kw_extractor.extract_keywords(block)
    kept = []
    for kw, score in keywords:
        words = kw.split()
        if len(words) == 2 and not any(w.lower() in IGNORER for w in words):
            kept.append(kw)
            compteur[kw.lower()] += 1
    print(f"Bloc {n} mentions these keywords: {', '.join(kept)}...")

Bloc 1 mentions these keywords: prince Léopold...
Bloc 2 mentions these keywords: gouvernement italien, Fédération Nationale, affaires étrangères, Comité International...
Bloc 3 mentions these keywords: ...
Bloc 4 mentions these keywords: VENTE PUBLIQUE, gare Nord, grand jardin...
Bloc 5 mentions these keywords: grand choix, Grand Cross...
Bloc 6 mentions these keywords: CENTRALE IMMOBILIERE...
Bloc 7 mentions these keywords: Pierre David, juge d’instruction, Louis XIII, Cour d’assises, mystérieusement disparu, Isidore Hacha, Mac Cord, cours d’une, Pierre Gif...
Bloc 8 mentions these keywords: gouvernement belge, cabinet Cuno, gouvernement Cuno, gouvernement français, contributions directes, réponse belge, Lloyd George...
Bloc 9 mentions these keywords: ...
Bloc 10 mentions these keywords: bat Mme, Union Liège, mètres match...
Bloc 11 mentions these keywords: ...
Bloc 12 mentions these keywords: ...
Bloc 13 mentions these keywords: Fédération nationale, Paris départ, l’institut Anti-Rh

In [25]:
compteur.most_common(20)

[('agence rossel', 11),
 ('rue van', 7),
 ('demande bon', 6),
 ('bons gages', 5),
 ('jeune fille', 5),
 ('demande bons', 5),
 ('demande jeune', 4),
 ('belle maison', 4),
 ('rue neuve', 4),
 ('jeune homme', 3),
 ('rue royale', 3),
 ('bons ouvriers', 3),
 ('louer maison', 3),
 ('l’agence rossel', 3),
 ('demande bonne', 3),
 ('fédération nationale', 2),
 ('demande place', 2),
 ('patisserie dem', 2),
 ('louer mais', 2),
 ('fille dem', 2)]